In [1]:
import pandas as pd

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv("https://raw.githubusercontent.com/hovhannisyan91/data_analytics_with_python/refs/heads/main/data/regression/logistic_regression/synthetic_churn_data.csv")
print(f"Dataset shape: {df.shape}")
df.head()


# df.to_csv("../../lab/python/data/regression/logistic_regression/synthetic_churn_data.csv", index=False)

Dataset shape: (1000, 6)


,tenure,monthly_charges,support_calls,contract_type,churn,education_level
0,29,46.867736,6,0,0,High School
1,15,74.163421,6,0,0,Bachelor
2,8,83.347822,3,1,0,Master
3,21,45.788769,2,0,0,High School
4,19,33.935607,7,0,1,Master


In [5]:
print(df.columns)
df.info()

Index(['tenure', 'monthly_charges', 'support_calls', 'contract_type', 'churn',
       'education_level'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   tenure           1000 non-null   int64  
 1   monthly_charges  1000 non-null   float64
 2   support_calls    1000 non-null   int64  
 3   contract_type    1000 non-null   int64  
 4   churn            1000 non-null   int64  
 5   education_level  1000 non-null   str    
dtypes: float64(1), int64(4), str(1)
memory usage: 53.8 KB


In [4]:
df['churn'].value_counts(normalize=True)

churn
0    0.633
1    0.367
Name: proportion, dtype: float64

In [6]:
churn_rate_by_contract = df.groupby('contract_type')['churn'].mean().reset_index()
import plotly.express as px
fig = px.bar(churn_rate_by_contract, x='contract_type', y='churn', title='Churn Rate by Contract Type')
fig.update_layout(
    xaxis_title='Contract Type (0 = Prepaid, 1 = Postpaid',
    yaxis_title='Churn Rate',
    template='plotly_white'
)   
fig.show()

In [7]:
X = df.drop(columns=["churn"])
y = df["churn"]

In [8]:
education_order = ["High School", "Bachelor", "Master", "PhD"]

df["education_level"] = pd.Categorical(
    df["education_level"],
    categories=education_order,
    ordered=True
)

education_dummies = pd.get_dummies(
    df["education_level"],
    prefix="education",
    drop_first=True,
    dtype=int
)

education_dummies.head()

,education_Bachelor,education_Master,education_PhD
0,0,0,0
1,1,0,0
2,0,1,0
3,0,0,0
4,0,1,0


In [9]:
X_encoded = pd.concat([X.drop(columns=["education_level"]), education_dummies], axis=1)

X_encoded.head()

,tenure,monthly_charges,support_calls,contract_type,education_Bachelor,education_Master,education_PhD
0,29,46.867736,6,0,0,0,0
1,15,74.163421,6,0,1,0,0
2,8,83.347822,3,1,0,1,0
3,21,45.788769,2,0,0,0,0
4,19,33.935607,7,0,0,1,0


In [10]:
print("Final set of features:")
print(X_encoded.dtypes)

Final set of features:
tenure                  int64
monthly_charges       float64
support_calls           int64
contract_type           int64
education_Bachelor      int64
education_Master        int64
education_PhD           int64
dtype: object


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [12]:
random_state=42

In [13]:
X_train, X_test, y_train, y_test

(     tenure  monthly_charges  support_calls  contract_type  \
 494      32        51.615610              6              0   
 687      26        52.433375              5              0   
 720      31        32.771348              2              0   
 381      33       100.172808              0              0   
 3        21        45.788769              2              0   
 ..      ...              ...            ...            ...   
 139      24        93.758230              8              1   
 565       3        57.325179              4              1   
 503      14       106.791829              9              0   
 759      20       106.201448              0              1   
 551      22        96.228513              0              0   
 
      education_Bachelor  education_Master  education_PhD  
 494                   1                 0              0  
 687                   0                 0              0  
 720                   1                 0              0  
 3

In [14]:
train_size = len(X_train)
test_size = len(X_test)

train_churn_rate = y_train.mean()
test_churn_rate = y_test.mean()

print("Training rows:", train_size)
print("Test rows:", test_size)
print("Training churn rate:", round(train_churn_rate, 3))
print("Test churn rate:", round(test_churn_rate, 3))

Training rows: 800
Test rows: 200
Training churn rate: 0.368
Test churn rate: 0.365


In [15]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [16]:
probs = model.predict_proba(X_test)
probs[:5]

array([[0.98318892, 0.01681108],
       [0.58272399, 0.41727601],
       [0.52510946, 0.47489054],
       [0.38401511, 0.61598489],
       [0.67937155, 0.32062845]])

In [17]:
probs = probs[:, 1]
probs[:5]

array([0.01681108, 0.41727601, 0.47489054, 0.61598489, 0.32062845])

In [18]:
threshold = 0.5
preds = (probs >= threshold).astype(int)
preds[:5]

array([0, 0, 0, 1, 0])

In [19]:
prediction_results = X_test.copy()

prediction_results["actual_churn"] = y_test.values
prediction_results["predicted_probability"] = probs
prediction_results["predicted_churn"] = preds

prediction_results.head()

,tenure,monthly_charges,support_calls,contract_type,education_Bachelor,education_Master,education_PhD,actual_churn,predicted_probability,predicted_churn
361,34,55.062693,2,1,0,0,0,0,0.016811,0
5,23,103.493024,4,0,0,0,1,0,0.417276,0
692,25,75.778342,5,0,0,1,0,1,0.474891,0
708,6,112.717782,5,1,0,1,0,0,0.615985,1
841,26,67.601813,7,1,0,0,1,0,0.320628,0
